<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week2/lab2/WPI_week2_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer-Based ECG Sequence Labeling

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 90-120 minutes.

## Learning Objectives

By the end of this lab, you should be able to:

- Describe how sequence labeling differs from whole-window classification.
- Build a compact Transformer encoder for ECG mask prediction.
- Explain the role of sequence-unit framing, positional information, and attention.
- Compare a context-aware model with a local convolutional baseline.

## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week2_lab2_responses_LastName_FirstName.docx`

Write Q1-Q5 in the Word document, using 2-5 sentences per question. Keep longer written responses in the Word document rather than in notebook markdown cells.

## Workflow

This lab reuses the ECG mask dataset from Lab 1, then changes the model family. You will train a compact CNN baseline and a Transformer sequence labeler, compare metrics, and inspect prediction plots.

## Setup

Run this first to load the scientific Python stack, the course helper package, and PyTorch.

In [ ]:
#@title Setup course environment
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("XDG_CACHE_HOME", "/tmp/.cache")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scipy",
    "pooch",
    "numpy",
    "pandas",
    "matplotlib",
])

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
    ])
    import torch

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    local_repo = Path.cwd()
    repo_dir = local_repo if (local_repo / "wpi_ai_bootcamp").exists() else Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])
elif (repo_dir / ".git").exists() and repo_dir.name == "WPI-AI-Course":
    subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])

sys.path.insert(0, str(repo_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

from wpi_ai_bootcamp.data import make_ecg_mask_dataset
from wpi_ai_bootcamp.notebook import setup_lab

WPI_COLORS = setup_lab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Setup complete. Device:", DEVICE)

## Data Loading

The dataset is generated at runtime from the public SciPy ECG sample. No local data file is required. The target is a per-sample R-peak mask, which makes this a sequence labeling task.

In [ ]:
X, y_mask, metadata = make_ecg_mask_dataset(
    window_s=2.0,
    stride_s=0.5,
    mask_radius_s=0.06,
    max_windows=96,
    random_state=7,
)

fs = metadata["fs"]
source = metadata["source"]
time = np.arange(X.shape[-1]) / fs

print("X shape:", X.shape)
print("mask shape:", y_mask.shape)
print("sampling rate:", fs, "Hz")
print("source:", source.name)
print("source url:", source.url)
print("positive mask fraction:", round(float(y_mask.mean()), 4))

source_table = pd.DataFrame([
    {
        "name": source.name,
        "loader": source.loader,
        "citation": source.citation,
        "license note": source.license_note,
    }
])
source_table

## Part 1 — Sequence Labels

A Transformer sees the ECG as an ordered sequence. Before modeling, inspect one window and think about which parts of the signal require local pattern recognition and which might benefit from broader context.

In [ ]:
def shade_mask_regions(ax, time_values, mask_values, *, label="Target mask", color=None, alpha=0.22):
    """Shade full-height time regions where a binary mask is active."""
    color = color or WPI_COLORS["accent_green"]
    time_values = np.asarray(time_values)
    active = np.asarray(mask_values) >= 0.5
    if not np.any(active):
        return

    edges = np.diff(active.astype(int), prepend=0, append=0)
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    dt = float(np.median(np.diff(time_values))) if len(time_values) > 1 else 0.0
    for region_index, (start, stop) in enumerate(zip(starts, stops)):
        ax.axvspan(
            time_values[start],
            time_values[min(stop, len(time_values) - 1)] + dt,
            color=color,
            alpha=alpha,
            label=label if region_index == 0 else None,
            zorder=0,
        )

# TODO: Change EXAMPLE_INDEX and compare a busy window with a quieter one.
EXAMPLE_INDEX = 3

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time, X[EXAMPLE_INDEX, 0], color=WPI_COLORS["black"], linewidth=1.0, label="ECG window")
shade_mask_regions(ax, time, y_mask[EXAMPLE_INDEX], label="Target mask", color=WPI_COLORS["accent_green"], alpha=0.20)
ax.set(title=f"Sequence labeling example {EXAMPLE_INDEX}", xlabel="Time within window (s)", ylabel="Normalized amplitude")
ax.legend(loc="upper right")
plt.show()

### Part 1 Assessment — Sequence Framing (15 pts)

- Notebook artifact: one ECG window with the mask overlaid. **8 pts**
- Word response Q1: Why is this lab framed as sequence labeling rather than classification? **7 pts**

## Part 2 — Data Split And Shared Training Helpers

We use the same split and metrics for both models so the comparison is easier to interpret.

In [ ]:
class ECGMaskDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

rng = np.random.default_rng(11)
indices = rng.permutation(len(X))
n_train = int(0.70 * len(indices))
n_val = int(0.15 * len(indices))
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

dataset = ECGMaskDataset(X, y_mask)

# TODO: Try a different batch size after you get the baseline run working.
BATCH_SIZE = 16

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=BATCH_SIZE)

print("train/val/test windows:", len(train_idx), len(val_idx), len(test_idx))

In [ ]:
def mask_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = probs >= threshold
    truth = targets >= 0.5
    tp = (preds & truth).sum().item()
    fp = (preds & ~truth).sum().item()
    fn = (~preds & truth).sum().item()
    union = (preds | truth).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    iou = tp / (union + 1e-8)
    return {"precision": precision, "recall": recall, "iou": iou}


def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    collected = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            total_loss += loss.item() * len(xb)
            collected.append(mask_metrics(logits.cpu(), yb.cpu()))
    metrics = pd.DataFrame(collected).mean().to_dict()
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics

## Part 3 — Local CNN Baseline

The baseline is intentionally small. It gives you a local pattern model to compare against the Transformer encoder.

In [ ]:
class LocalCNNBaseline(nn.Module):
    def __init__(self, hidden_channels=24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, hidden_channels, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, hidden_channels, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, 1, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

baseline_model = LocalCNNBaseline().to(DEVICE)
positive_fraction = float(y_mask[train_idx].mean())
pos_weight = torch.tensor([(1.0 - positive_fraction) / max(positive_fraction, 1e-6)], device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)

# TODO: Keep this short for a baseline, then decide whether the Transformer earns extra cost.
BASELINE_EPOCHS = 3
baseline_history = []
for epoch in range(1, BASELINE_EPOCHS + 1):
    train_loss = train_one_epoch(baseline_model, train_loader, optimizer, loss_fn)
    val_metrics = evaluate(baseline_model, val_loader, loss_fn)
    baseline_history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}})

pd.DataFrame(baseline_history)

## Part 4 — Transformer Sequence Labeler

The Transformer uses a strided convolution to turn the signal into shorter sequence units, adds positional information, and predicts a mask probability for each original time sample after interpolation.

In [ ]:
class TransformerMaskSegmenter(nn.Module):
    def __init__(self, d_model=48, nhead=4, num_layers=2, stride=4, max_units=256):
        super().__init__()
        self.stride = stride
        self.patch_embed = nn.Conv1d(1, d_model, kernel_size=9, stride=stride, padding=4)
        self.position = nn.Parameter(torch.zeros(1, max_units, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=2 * d_model,
            dropout=0.10,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        sequence_units = self.patch_embed(x).transpose(1, 2)
        sequence_units = sequence_units + self.position[:, : sequence_units.shape[1], :]
        encoded = self.encoder(sequence_units)
        low_res_logits = self.head(encoded).squeeze(-1)
        logits = F.interpolate(
            low_res_logits.unsqueeze(1),
            size=x.shape[-1],
            mode="linear",
            align_corners=False,
        ).squeeze(1)
        return logits

# TODO: Try one smaller or larger d_model value if runtime allows.
transformer_model = TransformerMaskSegmenter(d_model=48, nhead=4, num_layers=2).to(DEVICE)
print(transformer_model)
print("trainable parameters:", sum(p.numel() for p in transformer_model.parameters()))

In [ ]:
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-3)

# TODO: Increase EPOCHS by 1-2 if validation metrics are still improving.
EPOCHS = 4
transformer_history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(transformer_model, train_loader, optimizer, loss_fn)
    val_metrics = evaluate(transformer_model, val_loader, loss_fn)
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    transformer_history.append(row)
    print(row)

pd.DataFrame(transformer_history)

### Part 4 Assessment — Transformer Training (30 pts)

- Notebook artifact: Transformer model summary and training history. **18 pts**
- Word response Q2: What does the patch_embed stride do to sequence length and runtime? **6 pts**
- Word response Q3: Why does the model need positional information after sequence-unit framing? **6 pts**

## Part 5 — Compare Models

Use the same held-out test windows for both models. A higher score is useful, but the prediction plot matters too because segmentation errors can be clinically or scientifically different even when summary metrics look similar.

In [ ]:
comparison = pd.DataFrame([
    {"model": "local CNN baseline", **evaluate(baseline_model, test_loader, loss_fn)},
    {"model": "Transformer encoder", **evaluate(transformer_model, test_loader, loss_fn)},
])
comparison

In [ ]:
def plot_model_comparison(models, dataset, sample_position=0):
    xb, yb = dataset[sample_position]
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time, xb.squeeze(0).numpy(), color=WPI_COLORS["black"], linewidth=1.0, label="ECG window")
    shade_mask_regions(ax, time, yb.numpy(), label="Target mask", color=WPI_COLORS["accent_green"], alpha=0.20)

    for name, model, color in models:
        model.eval()
        with torch.no_grad():
            logits = model(xb.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)
        probs = torch.sigmoid(logits).numpy()
        ax.plot(time, probs, linewidth=1.4, color=color, label=name)

    ax.set(title=f"Model comparison on test sample {sample_position}", xlabel="Time within window (s)", ylabel="Normalized signal / probability")
    ax.legend(loc="upper right")
    plt.show()

# TODO: Inspect at least two sample positions before answering Q4.
SAMPLE_POSITION = 0
plot_model_comparison(
    [
        ("CNN probability", baseline_model, WPI_COLORS["accent_blue"]),
        ("Transformer probability", transformer_model, WPI_COLORS["crimson"]),
    ],
    Subset(dataset, test_idx),
    SAMPLE_POSITION,
)

### Part 5 Assessment — Model Comparison (35 pts)

- Notebook artifact: test comparison table and at least one model comparison plot. **20 pts**
- Word response Q4: Which model performed better on your run, and what evidence supports that claim? **8 pts**
- Word response Q5: Name one reason the Transformer could help on longer signals and one reason it may be unnecessary for this short-window lab. **7 pts**

TODO: Include the sample position and the metric you relied on most.

## Optional Challenge

TODO: Change the Transformer `stride` from `4` to `2` or `8`. Explain how this changes sequence unit count, runtime, and the smoothness of the predicted mask.

## Attribution

- Data: SciPy electrocardiogram sample, loaded through `scipy.datasets.electrocardiogram()` via `wpi_ai_bootcamp.data.make_ecg_mask_dataset()`.
- Source note: SciPy documents the sample as derived from MIT-BIH Arrhythmia Database record 208 on PhysioNet.
- Libraries: NumPy, pandas, Matplotlib, SciPy, and PyTorch.
- WPI colors: WPI Institutional Guidelines, Crimson `#AC2B37` and Gray `#A9B0B7`.